In [1]:
from transformers import AutoTokenizer

In [2]:
prompt = "It was a dark and stormy"

In [3]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B")

In [4]:
input_ids = tokenizer(prompt).input_ids

In [5]:
input_ids

[2132, 572, 264, 6319, 323, 13458, 88]

In [6]:
for t in input_ids:
    print(t,"\t",tokenizer.decode(t))

2132 	 It
572 	  was
264 	  a
6319 	  dark
323 	  and
13458 	  storm
88 	 y


In [7]:
from transformers import AutoModelForCausalLM

In [8]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B")
#We tokenize again but specifying the tokenizer that we want it to return apytorch tensor, which is what the model expects,rather than a list of ints.
input_ids = tokenizer(prompt,return_tensors="pt").input_ids
outputs = model(input_ids)
outputs.logits.shape

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

torch.Size([1, 7, 151936])

In [9]:
final_logits = model(input_ids).logits[0,-1] #The last set of logits
final_logits.argmax() #The position of the maximum

tensor(3729)

In [10]:
tokenizer.decode(final_logits.argmax())

' night'

In [11]:
#Lets now find out which other tokes were potential candidates by selecting the top 10 values with topk()

In [12]:
import torch
top10_logits = torch.topk(final_logits,10)
for index in top10_logits.indices:
    print(tokenizer.decode(index))

 night
 evening
 day
 morning
 winter
 afternoon
 Saturday
 Sunday
 Friday
 Monday


In [13]:
#now lets convert logits to probabilities 

In [14]:
top10 = torch.topk(final_logits.softmax(dim=0),10)
for value, index in zip(top10.values, top10.indices):
    print(f"{tokenizer.decode(index):<10} {value.item():.2%}")

 night     88.67%
 evening   4.39%
 day       2.36%
 morning   0.46%
 winter    0.44%
 afternoon 0.28%
 Saturday  0.25%
 Sunday    0.19%
 Friday    0.18%
 Monday    0.16%


## Generating Text

In [16]:
#Greedy decoding

In [17]:
output_ids = model.generate(input_ids, max_new_tokens=20)
decoded_text = tokenizer.decode(output_ids[0])

print("Input IDs",input_ids[0])
print("Output IDs",output_ids)
print(f"Generated text: {decoded_text}")

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Input IDs tensor([ 2132,   572,   264,  6319,   323, 13458,    88])
Output IDs tensor([[ 2132,   572,   264,  6319,   323, 13458,    88,  3729,    13,   576,
         12884,   572,  6319,   323,   279,  9956,   572,  1246,  2718,    13,
           576, 11174,   572, 50413,  1495,   323,   279]])
Generated text: It was a dark and stormy night. The sky was dark and the wind was howling. The rain was pouring down and the


In [18]:
#Beam Search

In [19]:
beam_output = model.generate(
    input_ids,
    num_beams=5,
    max_new_tokens=30,
)
print(tokenizer.decode(beam_output[0]))

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy night. The wind was howling, and the rain was pouring down. The sky was dark and gloomy, and the air was filled with the


In [21]:
beam_output = model.generate(
    input_ids,
    num_beams=5,
    repetition_penalty = 2.0,
    max_new_tokens=38,
)
print(tokenizer.decode(beam_output[0]))

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy night. The wind howled, the rain pounded, and the thunder rumbled through the air. It was just after 10:00 p.m., and I was sitting in


In [22]:
#Sampling

In [23]:
from transformers import set_seed
set_seed(70)

sampling_output = model.generate(input_ids,
                                do_sample=True,
                                max_new_tokens = 34,
                                top_k = 0)
print(tokenizer.decode(sampling_output[0]))

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy night, the temperature in the neighborhood was a hundred and eighty degrees.
Our News Minute Tonight shared our voice, on What It Takes to Win Heartbreak on Newscast


In [24]:
sampling_output = model.generate(input_ids,
                                do_sample=True,
                                temperature = 0.001,
                                max_new_tokens = 34,
                                top_k = 0)
print(tokenizer.decode(sampling_output[0]))

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy night. The sky was dark and the wind was howling. The rain was pouring down and the lightning was flashing. The sky was dark and the wind was howling


In [25]:
sampling_output = model.generate(input_ids,
                                do_sample=True,
                                temperature = 0.4,
                                max_new_tokens = 34,
                                top_k = 0)
print(tokenizer.decode(sampling_output[0]))

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy night in the small town of Waco, Texas. The sky was dark and the wind howled. The streets were filled with people, and the cars were riddled


In [28]:
sampling_output = model.generate(input_ids,
                                do_sample=True,
                                temperature = 2.0,
                                max_new_tokens = 34,
                                top_k = 0)
print(tokenizer.decode(sampling_output[0]))

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy SummerViewSet jQuery持有人eed mudmask Whites...

เมนูeecৎ느 exhilar swords seasHe Bd HibernateOthers Турية Find deploy Exhibition strtotimering finishing invadingmarker…
Щ Uniform


In [29]:
sampling_output = model.generate(input_ids,
                                do_sample=True,
                                max_new_tokens = 34,
                                top_k = 5)
print(tokenizer.decode(sampling_output[0]))

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy night in the city of Los Angeles. The streets were quiet and the only sounds were the wind, the rain, and the howling of the storm that was about to


In [31]:
sampling_output = model.generate(input_ids,
                                do_sample=True,
                                max_new_tokens = 34,
                                top_k = 0,
                                top_p = 0.94)
print(tokenizer.decode(sampling_output[0]))

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy night in those least soggy dregs of Michigan — the Wobegon that just happened to have adjectives — that an Austrian builder and importer of rough-wear
